In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text(
    "environment",
    "dev",
    "Environment"
)

environment = dbutils.widgets.get("environment").lower()

if environment not in ["dev", "prod"]:
    raise ValueError(
        "Environment must be either 'dev' or 'prod'."
    )

config = {
    "dev": {
        "catalog": "saleslt_dev"
    },
    "prod": {
        "catalog": "saleslt_prod"
    }
}

catalog = config[environment]["catalog"]

# Bronze
customer_bronze = f"{catalog}.bronze.customer_raw"
product_bronze = f"{catalog}.bronze.product_raw"
category_bronze = f"{catalog}.bronze.product_category_raw"
header_bronze = f"{catalog}.bronze.sales_order_header_raw"
detail_bronze = f"{catalog}.bronze.sales_order_detail_raw"

# Silver
customer_silver = f"{catalog}.silver.customers"
product_silver = f"{catalog}.silver.products"
sales_silver = f"{catalog}.silver.sales_order_lines"

# Gold
product_gold = f"{catalog}.gold.sales_by_product"
customer_gold = f"{catalog}.gold.sales_by_customer"
monthly_gold = f"{catalog}.gold.monthly_sales_summary"

print("=" * 60)
print("SALESLT - METADATA DOCUMENTATION")
print("=" * 60)
print(f"Environment : {environment}")
print(f"Catalog     : {catalog}")
print("=" * 60)

In [0]:

def apply_column_comments(table_name, comments):

    existing_columns = {
        field.name
        for field in spark.table(table_name).schema.fields
    }

    for column_name, comment in comments.items():

        if column_name in existing_columns:

            spark.sql(f"""
                ALTER TABLE {table_name}
                ALTER COLUMN `{column_name}`
                COMMENT '{comment}'
            """)

    print(
        f"Column comments applied: {table_name}"
    )

In [0]:

bronze_table_comments = {
    customer_bronze:
        "Raw customer data replicated from Azure SQL SalesLT through Lakehouse Federation. "
        "The table preserves source attributes and adds technical ingestion metadata.",

    product_bronze:
        "Raw product data replicated from Azure SQL SalesLT through Lakehouse Federation. "
        "The table preserves source product attributes for downstream transformation.",

    category_bronze:
        "Raw SalesLT product-category reference data replicated from Azure SQL through Lakehouse Federation.",

    header_bronze:
        "Raw SalesLT sales order header data replicated from Azure SQL through Lakehouse Federation. "
        "Contains order-level dates, customer references, taxes, freight, and total values.",

    detail_bronze:
        "Raw SalesLT sales order detail data replicated from Azure SQL through Lakehouse Federation. "
        "Contains product-level order quantities, pricing, discounts, and line totals."
}

for table_name, comment in bronze_table_comments.items():

    spark.sql(f"""
        COMMENT ON TABLE {table_name}
        IS '{comment}'
    """)

In [0]:


apply_column_comments(
    header_bronze,
    {
        "SalesOrderID":
            "Unique identifier of the SalesLT sales order.",

        "CustomerID":
            "Identifier of the customer associated with the sales order.",

        "OrderDate":
            "Timestamp when the sales order was created.",

        "SubTotal":
            "Sales order subtotal before taxes and freight.",

        "TaxAmt":
            "Tax amount applied to the sales order.",

        "Freight":
            "Freight amount applied to the sales order.",

        "TotalDue":
            "Total order amount including taxes and freight.",

        "ingestion_timestamp":
            "Timestamp associated with the Bronze replication execution."
    }
)

apply_column_comments(
    detail_bronze,
    {
        "SalesOrderDetailID":
            "Unique identifier of the sales order line.",

        "SalesOrderID":
            "Identifier of the parent sales order.",

        "ProductID":
            "Identifier of the product sold on the order line.",

        "OrderQty":
            "Quantity of product units ordered.",

        "UnitPrice":
            "Unit selling price provided by the SalesLT source.",

        "UnitPriceDiscount":
            "Discount percentage applied to the order line.",

        "LineTotal":
            "Source-calculated monetary total for the order line.",

        "ingestion_timestamp":
            "Timestamp associated with the Bronze replication execution."
    }
)

In [0]:

silver_table_comments = {
    customer_silver:
        "Cleaned and standardized SalesLT customer dimension containing normalized names, "
        "contact information, and customer attributes for downstream analytics.",

    product_silver:
        "Cleaned and enriched SalesLT product dimension joined with product-category metadata. "
        "Contains standardized pricing, product attributes, and active-status information.",

    sales_silver:
        "Analytical SalesLT sales-order-line dataset produced by joining sales order headers, "
        "order details, customers, products, and product categories. Monetary values are "
        "normalized to two-decimal precision for analytics."
}

for table_name, comment in silver_table_comments.items():

    spark.sql(f"""
        COMMENT ON TABLE {table_name}
        IS '{comment}'
    """)

In [0]:
apply_column_comments(
    customer_silver,
    {
        "customer_id":
            "Unique identifier of the customer.",

        "full_name":
            "Standardized full customer name assembled from source name components.",

        "company_name":
            "Company associated with the customer when provided.",

        "email_address":
            "Normalized customer email address.",

        "phone":
            "Customer telephone number.",

        "source_modified_timestamp":
            "Last modification timestamp provided by the SalesLT source.",

        "silver_processing_timestamp":
            "Timestamp associated with the Silver transformation execution."
    }
)

apply_column_comments(
    product_silver,
    {
        "product_id":
            "Unique identifier of the product.",

        "product_name":
            "Standardized product name.",

        "product_number":
            "Business product number from SalesLT.",

        "standard_cost":
            "Normalized product standard cost.",

        "list_price":
            "Normalized product list price.",

        "product_category":
            "Product category enriched from the SalesLT ProductCategory source.",

        "is_active":
            "Indicates whether the product is currently active based on discontinuation metadata.",

        "silver_processing_timestamp":
            "Timestamp associated with the Silver transformation execution."
    }
)

apply_column_comments(
    sales_silver,
    {
        "sales_order_detail_id":
            "Unique identifier of the analytical sales order line.",

        "sales_order_id":
            "Identifier of the parent sales order.",

        "sales_order_number":
            "Business sales order number.",

        "customer_id":
            "Identifier of the customer associated with the order.",

        "customer_name":
            "Standardized customer name enriched from the customer source.",

        "customer_email":
            "Normalized customer email address used for customer analytics.",

        "product_id":
            "Identifier of the product sold.",

        "product_name":
            "Product name enriched from SalesLT product data.",

        "product_category":
            "Product category enriched from SalesLT product-category data.",

        "order_date":
            "Timestamp when the sales order was created.",

        "order_quantity":
            "Quantity of product units sold on the order line.",

        "unit_price":
            "Unit selling price normalized to two-decimal monetary precision.",

        "unit_price_discount":
            "Discount percentage applied to the order line.",

        "gross_line_amount":
            "Order-line value before discounts.",

        "discount_amount":
            "Calculated monetary discount applied to the order line.",

        "net_line_amount":
            "Order-line revenue after discount, normalized to two-decimal precision.",

        "order_total_due":
            "Total amount due for the complete sales order.",

        "silver_processing_timestamp":
            "Timestamp associated with the Silver transformation execution."
    }
)


In [0]:
gold_table_comments = {
    product_gold:
        "Sales performance aggregated by product, including order counts, customer counts, "
        "units sold, revenue, discounts, order-value statistics, and revenue ranking. "
        "Designed for product analytics, Databricks Genie, dashboards, and BI.",

    customer_gold:
        "Customer sales performance summary containing order activity, product diversity, "
        "units purchased, revenue, discounts, spending metrics, and customer activity dates. "
        "Designed for customer analytics and governed BI consumption.",

    monthly_gold:
        "Monthly SalesLT sales performance summary containing orders, customers, products, "
        "units sold, revenue, discounts, and order-value statistics for time-series analytics."
}

for table_name, comment in gold_table_comments.items():

    spark.sql(f"""
        COMMENT ON TABLE {table_name}
        IS '{comment}'
    """)

In [0]:
apply_column_comments(
    product_gold,
    {
        "product_id":
            "Unique identifier of the product.",

        "product_name":
            "Business name of the product.",

        "product_category":
            "Business category assigned to the product.",

        "total_orders":
            "Number of distinct sales orders containing the product.",

        "total_customers":
            "Number of distinct customers who purchased the product.",

        "total_units_sold":
            "Total quantity of product units sold.",

        "gross_revenue":
            "Total revenue before discounts.",

        "total_discount":
            "Total monetary discounts associated with the product.",

        "net_revenue":
            "Total product revenue after discounts.",

        "average_line_value":
            "Average net value of a sales order line containing the product.",

        "revenue_rank":
            "Dense ranking of the product based on net revenue, where rank 1 represents the highest revenue.",

        "gold_processing_timestamp":
            "Timestamp associated with the Gold aggregation execution."
    }
)

apply_column_comments(
    customer_gold,
    {
        "customer_id":
            "Unique identifier of the customer.",

        "customer_name":
            "Standardized customer name.",

        "customer_email":
            "Customer email address. Intended for demonstration of governed column masking.",

        "total_orders":
            "Number of distinct orders placed by the customer.",

        "distinct_products":
            "Number of distinct products purchased by the customer.",

        "total_units_purchased":
            "Total number of product units purchased by the customer.",

        "gross_revenue":
            "Total customer sales amount before discounts.",

        "total_discount":
            "Total monetary discount applied to customer purchases.",

        "total_spent":
            "Total net revenue attributed to the customer.",

        "average_line_value":
            "Average net sales order-line value for the customer.",

        "first_order_date":
            "Earliest order timestamp represented in the customer summary.",

        "last_order_date":
            "Most recent order timestamp represented in the customer summary.",

        "gold_processing_timestamp":
            "Timestamp associated with the Gold aggregation execution."
    }
)

apply_column_comments(
    monthly_gold,
    {
        "sales_year":
            "Calendar year represented by the monthly sales aggregation.",

        "sales_month":
            "Calendar month number represented by the aggregation.",

        "month_start_date":
            "First timestamp of the reporting month.",

        "total_orders":
            "Number of distinct sales orders during the month.",

        "total_customers":
            "Number of distinct customers purchasing during the month.",

        "distinct_products":
            "Number of distinct products sold during the month.",

        "total_units_sold":
            "Total product units sold during the month.",

        "gross_revenue":
            "Monthly revenue before discounts.",

        "total_discount":
            "Monthly monetary discounts.",

        "net_revenue":
            "Monthly revenue after discounts.",

        "average_line_value":
            "Average net sales order-line value during the month.",

        "min_line_value":
            "Lowest net sales order-line value during the month.",

        "max_line_value":
            "Highest net sales order-line value during the month.",

        "gold_processing_timestamp":
            "Timestamp associated with the Gold aggregation execution."
    }
)
